# BRKGA Foco — Flow Shop Scheduling

Experimentos com o brkga_scs

## 0. Setup — Instalação e Imports

In [1]:
# Instala dependencias
!pip install -q pymoo scikit-fuzzy optuna numpy pandas
#!pip install -q torch --index-url https://download.pytorch.org/whl/cpu
print('Dependencias instaladas.')

Dependencias instaladas.


In [2]:
# Monta Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
ROOT = '/content/drive/MyDrive/FlowShopScheduling'

import sys, os, time, json, csv
sys.path.insert(0, ROOT)

from pathlib import Path
from typing import List, Dict, Optional, Tuple, Callable

import numpy as np
import pandas as pd
import optuna
import json

DATA_DIR = Path(ROOT) / 'data' / 'taillard'
RESULTADOS_DIR = Path(ROOT) / 'resultados_brkga_scs'
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Resultados em: {RESULTADOS_DIR}')

ROOT: /content/drive/MyDrive/FlowShopScheduling
DATA_DIR: /content/drive/MyDrive/FlowShopScheduling/data/taillard
Resultados em: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_scs


In [5]:
# Imports do projeto
from src.flowshop_engine import FlowShopEngine
from src.solvers.adapter import rodar_brkga_scs_vm, rodar_brkga_scs_vr, SolverResult

print('Imports concluídos.')

Imports concluídos.


## 1. Catálogo de Instâncias — Small and Taillard

Extrai todas as instâncias com 16 jobs do diretório `data/Small/`.
Parseia o nome do arquivo: `I_{grupo}_{n_jobs}_{n_machines}_{id}.txt`.

In [6]:
def construir_catalogo_taillard(data_dir: Path) -> pd.DataFrame:
    """
    Lê os arquivos ta*.txt, extrai as dimensões e retorna um DataFrame do catálogo.
    """
    # Encontra todos os arquivos que começam com "ta" e terminam com ".txt"
    todos = sorted(data_dir.glob('ta*.txt'))
    registros = []

    for f in todos:
        # Extrai o ID numérico (ex: "ta005.txt" -> 5)
        id_instancia = int(f.stem.replace('ta', ''))

        # Dá uma "espiada" rápida no arquivo para extrair jobs e machines da Linha 2
        with open(f, 'r') as file:
            linhas = [l.strip() for l in file if l.strip()]
            partes = linhas[1].split()
            n_jobs = int(partes[0])
            n_machines = int(partes[1])
        bks = int(partes[3]) # Já extrai o BKS para termos na tabela!

        registros.append({
            'arquivo': f.name,
            'id': id_instancia,
            'n_jobs': n_jobs,
            'n_machines': n_machines,
            'bks': bks,
            'path': str(f),
            'rotulo': f'{n_jobs}x{n_machines}_id{id_instancia}'
        })

    df = pd.DataFrame(registros)
    df = df.sort_values(['n_jobs', 'n_machines', 'id']).reset_index(drop=True)
    return df



#MUDAR AQUI SE EU QUISER TESTAR OUTRO CATALOGO
catalogo = construir_catalogo_taillard(DATA_DIR)

print(f'Total de instâncias: {len(catalogo)}')
print()
print('Distribuição por tamanho (Jobs x Máquinas):')
# Agrupa para mostrar quantos arquivos temos de cada tamanho
distribuicao = catalogo.groupby(['n_jobs', 'n_machines']).size()
for (j, m), qtd in distribuicao.items():
    print(f'  {j}x{m}: {qtd} instâncias')

print('\n10 primeiras instâncias:')
print(catalogo.head(10).to_string(index=False))

Total de instâncias: 120

Distribuição por tamanho (Jobs x Máquinas):
  20x5: 10 instâncias
  20x10: 10 instâncias
  20x20: 10 instâncias
  50x5: 10 instâncias
  50x10: 10 instâncias
  50x20: 10 instâncias
  100x5: 10 instâncias
  100x10: 10 instâncias
  100x20: 10 instâncias
  200x10: 10 instâncias
  200x20: 10 instâncias
  500x20: 10 instâncias

10 primeiras instâncias:
  arquivo  id  n_jobs  n_machines  bks                                                              path    rotulo
ta001.txt   1      20           5 1278 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta001.txt  20x5_id1
ta002.txt   2      20           5 1359 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta002.txt  20x5_id2
ta003.txt   3      20           5 1081 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta003.txt  20x5_id3
ta004.txt   4      20           5 1293 /content/drive/MyDrive/FlowShopScheduling/data/taillard/ta004.txt  20x5_id4
ta005.txt   5      20           5 1236 /content/dr

In [7]:
POP_SIZE = 100

# Geracoes para o experimento comparativo (se eu passar o t_param, automaticamente esse nao vai se ativo)
MAX_EVALS = 50000

# Sementes para o experimento
N_SEEDS = 5
SEED_INICIO = 100

# Objetivo
OBJECTIVE = 'Makespan'

## 5. Fábrica de Variantes (closures)

Cada variante é uma callable `(engine, seed) -> SolverResult`.

In [ ]:
""" Isso é para nao ter que ficar passando os paramtros toda vez que chamar os alg...."""
def fabricar_variante_brkga_scs_vm() -> Callable:
    """BRKGA-SCS (Clustering Search) com parametros padrao do notebook original."""
    def _run(engine, seed):
        return rodar_brkga_scs_vm(
            engine, seed=seed, max_evals=MAX_EVALS, objective=OBJECTIVE,
            t_param=60, # fator multiplicante para o tempo de cpu
            tam_pop=POP_SIZE, pe=0.2, pm=0.15, rho=0.7,
            z_limiar=1.5, votos_limiar=5,
            num_clusters_max=5, r_min=0.05, r_max=0.25,
            base_cooldown=5, gamma=1.5,
        )
    return _run

def fabricar_variante_brkga_scs_vr() -> Callable:
    def _run(engine, seed):
        return rodar_brkga_scs_vr(
            engine, seed=seed, max_evals=MAX_EVALS,
            t_param=60, # ou None se quiser parar por max_evals
            tam_pop=POP_SIZE, pe=0.15, pm=0.25, rho=0.85,
            raio_cluster=0.5, z_limiar=1.645, votos_limiar=5,
            max_hj_evals=600,
        )
    return _run

# Monta o dicionario de variantes
variantes = {
    #'brkga_scs_vm': fabricar_variante_brkga_scs_vm(),
    'brkga_scs_vr': fabricar_variante_brkga_scs_vr(),
    
}

print(f'Variantes disponiveis: {list(variantes.keys())}')


Variantes disponiveis: ['brkga_scs_vr']


## 6. Loop Comparativo - Taillard


In [12]:
def amostrar_historico_v1(history: list, max_evals: int, passo: int = 2000) -> dict:
    """Extrai o makespan nos marcos evals=passo, 2*passo, ..., max_evals."""
    amostras = {}

    # history é lista de (evals, makespan)
    # se history estiver vazio, retorne NaN
    if not history:
        for ev in range(passo, max_evals + 1, passo):
            amostras[f'mks_ev{ev}'] = float('nan')
        return amostras

    for ev in range(passo, max_evals + 1, passo):
        # Encontra o ultimo makespan reportado ANTES ou NO marco ev
        best_mks = history[0][1]
        for e, m in history:
            if e <= ev:
                best_mks = m
            else:
                break
        amostras[f'mks_ev{ev}'] = float(best_mks)
    return amostras


# Schema fixo do CSV
CAMPOS_CSV = [
    'instancia', 'grupo', 'n_jobs', 'n_machines',
    'metodo', 'seed',
    'makespan', 'tardiness', 'cfo_count', 'tempo_s',
    'lower_bound', 'bks', 'rpd',
]

#Quiser alterar o arquivo csv de comparacao é aqui essa merda
CSV_PATH = RESULTADOS_DIR / 'results_brkga_scs_vr_taillard.csv'
print(f'CSV de saida: {CSV_PATH}')
print(f'Campos: {CAMPOS_CSV}')


CSV de saida: /content/drive/MyDrive/FlowShopScheduling/resultados_brkga_scs/results_brkga_scs_vr_taillard.csv
Campos: ['instancia', 'grupo', 'n_jobs', 'n_machines', 'metodo', 'seed', 'makespan', 'tardiness', 'cfo_count', 'tempo_s', 'lower_bound', 'bks', 'rpd']


In [13]:
def amostrar_historico(historico, max_evals, passo):
    amostras = {}
    for ev in range(passo, max_evals + 1, passo):
        amostras[f'mks_ev{ev}'] = float('nan')

    if not historico:
        return amostras

    alvos = list(range(passo, max_evals + 1, passo))
    idx_alvo = 0
    best_mks = historico[0][-1] # Pega sempre o último elemento do item (makespan)

    # Lendo o histórico de forma segura independente de ter 2 ou 3 colunas:
    for item in historico:
        n_evals = item[0]
        atual_mks = item[-1]

        while idx_alvo < len(alvos) and n_evals >= alvos[idx_alvo]:
            amostras[f'mks_ev{alvos[idx_alvo]}'] = float(best_mks)
            idx_alvo += 1

        best_mks = atual_mks

    while idx_alvo < len(alvos):
        amostras[f'mks_ev{alvos[idx_alvo]}'] = float(best_mks)
        idx_alvo += 1

    return amostras

In [14]:
print('=' * 72)
print('INICIANDO LOOP COMPARATIVO')
print(f'{len(catalogo)} instancias x {len(variantes)} variantes x {N_SEEDS} seeds')
print(f'Total: {len(catalogo) * len(variantes) * N_SEEDS} execucoes')
print('=' * 72)

t0_total = time.time()

with open(DATA_DIR / 'taillard_bks.json', 'r') as f:
    bks_db = json.load(f)

csv_header_escrito = False

for idx_inst, (_, row) in enumerate(catalogo.iterrows()):
    inst_label = row['rotulo']
    inst_path = row['path']
    n_jobs = row['n_jobs']
    n_mach = row['n_machines']
    #grupo = row['grupo']
    grupo = "Taillard"

    # Carrega engine uma vez por instancia
    nj, nm, pt, dd = FlowShopEngine.carregar_instancia_taillard(inst_path)
    engine = FlowShopEngine(nj, nm, pt, dd)

    for metodo, fn_variante in variantes.items():
        for seed in range(SEED_INICIO, SEED_INICIO + N_SEEDS):
            # LOG de progresso detalhado
            print(
                f'[{idx_inst + 1}/{len(catalogo)}] '
                f'Inst: {row["arquivo"]} ({n_jobs}j x {n_mach}m, g{grupo}) '
                f'| Var: {metodo} '
                f'| Seed: {seed} ...',
                end=' '
            )

            try:
                t0_exec = time.time()
                result = fn_variante(engine, seed)
                elapsed = time.time() - t0_exec

                # Monta linha do CSV
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float(result.best_cost),
                    'tardiness': float(result.tardiness),
                    'cfo_count': int(result.n_evaluations),
                    'tempo_s': round(elapsed, 3),
                    'lower_bound': float(result.lower_bound),
                }

                # Nova logica RPD
                bks_real = bks_db.get(row['arquivo'].replace('.txt', ''))
                if bks_real is not None:
                    rpd = ((result.best_cost - bks_real) / bks_real) * 100.0
                else:
                    rpd = float('nan')
                linha['bks'] = bks_real
                linha['rpd'] = round(rpd, 3)

                print(f'OK  Makespan: {result.best_cost:.1f} ({elapsed:.2f}s)')

            except Exception as e:
                print(f'ERRO: {e}')
                linha = {
                    'instancia': row['arquivo'],
                    'grupo': grupo,
                    'n_jobs': n_jobs,
                    'n_machines': n_mach,
                    'metodo': metodo,
                    'seed': seed,
                    'makespan': float('nan'),
                    'tardiness': float('nan'),
                    'cfo_count': 0,
                    'tempo_s': float('nan'),
                    'lower_bound': float('nan'),
                    'bks': float('nan'),
                    'rpd': float('nan'),
                }

            # Escreve no CSV incremental (abre/fecha a cada linha = seguro)
            escrever_header = not csv_header_escrito
            with open(CSV_PATH, 'w' if escrever_header else 'a', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=CAMPOS_CSV)
                if escrever_header:
                    writer.writeheader()
                    csv_header_escrito = True
                writer.writerow(linha)

elapsed_total = time.time() - t0_total
print('=' * 72)
print(f'LOOP CONCLUIDO em {elapsed_total:.1f}s ({elapsed_total/60:.1f}min)')
print(f'Resultados salvos em: {CSV_PATH}')


INICIANDO LOOP COMPARATIVO
120 instancias x 1 variantes x 5 seeds
Total: 600 execucoes
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 100 ... OK  Makespan: 1297.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 101 ... OK  Makespan: 1297.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 102 ... OK  Makespan: 1297.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 103 ... OK  Makespan: 1297.0 (3.01s)
[1/120] Inst: ta001.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 104 ... OK  Makespan: 1322.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 100 ... OK  Makespan: 1373.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 101 ... OK  Makespan: 1368.0 (3.01s)
[2/120] Inst: ta002.txt (20j x 5m, gTaillard) | Var: brkga_scs_vr | Seed: 102 ... OK  Makespan: 1367.0 (3.01s)
[2/120] Inst: ta002.txt (

KeyboardInterrupt: 

## 7. Resumo Final

In [15]:
df = pd.read_csv(CSV_PATH)
print(f'Total de linhas no CSV: {len(df)}')
print()

print('=' * 120)
print('TABELA COMPARATIVA — Makespan medio por Instancia x Metodo')
print('=' * 120)

# Tabela detalhada por instancia + metodo
agg = df.groupby(['instancia', 'n_jobs', 'n_machines', 'metodo']).agg(
    makespan_medio=('makespan', 'mean'),
    desvio=('makespan', 'std'),
    melhor=('makespan', 'min'),
    pior=('makespan', 'max'),
    tempo_medio=('tempo_s', 'mean'),
    bks = ('bks', 'first'),
    rpd_medio=('rpd', 'mean'),
    lower_bound=('lower_bound', 'first'),
).reset_index()
agg['makespan_medio'] = agg['makespan_medio'].round(1)
agg['desvio'] = agg['desvio'].round(1)
agg['tempo_medio'] = agg['tempo_medio'].round(3)
agg['rpd_medio'] = agg['rpd_medio'].round(2)

print(agg.to_string(index=False))
print()

# Resumo global (agregando todas as instancias)
global_agg = df.groupby('metodo').agg(
    makespan_medio=('makespan', 'mean'),
    desvio_std=('makespan', 'std'),
    melhor_global=('makespan', 'min'),
    tempo_medio=('tempo_s', 'mean'),
    rpd_medio=('rpd', 'mean'),
).reset_index()
global_agg['makespan_medio'] = global_agg['makespan_medio'].round(1)
global_agg['desvio_std'] = global_agg['desvio_std'].round(1)
global_agg['tempo_medio'] = global_agg['tempo_medio'].round(3)
global_agg['rpd_medio'] = global_agg['rpd_medio'].round(2)

print('-' * 80)
print('RESUMO GLOBAL — Makespan medio por Metodo')
print('-' * 80)
print(global_agg.to_string(index=False))
print()
print(f'Arquivo salvo em: {CSV_PATH}')

Total de linhas no CSV: 11

TABELA COMPARATIVA — Makespan medio por Instancia x Metodo
instancia  n_jobs  n_machines       metodo  makespan_medio  desvio  melhor   pior  tempo_medio  bks  rpd_medio  lower_bound
ta001.txt      20           5 brkga_scs_vr          1302.0    11.2  1297.0 1322.0        3.007 1278       1.88       1249.0
ta002.txt      20           5 brkga_scs_vr          1369.4     3.4  1366.0 1373.0        3.007 1359       0.77       1346.0
ta003.txt      20           5 brkga_scs_vr          1165.0     NaN  1165.0 1165.0        3.006 1081       7.77       1073.0

--------------------------------------------------------------------------------
RESUMO GLOBAL — Makespan medio por Metodo
--------------------------------------------------------------------------------
      metodo  makespan_medio  desvio_std  melhor_global  tempo_medio  rpd_medio
brkga_scs_vr          1320.2        62.0         1165.0        3.007       1.91

Arquivo salvo em: /content/drive/MyDrive/FlowShopSc